# Evaluacion cuantitativa del RAG



Hasta este momento hemos realizado un agente que parece estar desenvolviendose correctamente. Sin embargo no podemos quedarnos con sensaciones, somos ingenieros y tenemos que medir todo, ya que lo que no se puede medir, no se puede gestionar, y lo que no se puede gestionar, no se puede mejorar

Es importante aclarar que nos vamos a remitir a evaluar unicamente el modulo de `Retrieval` del **RAG**, ya qu eintentar medir todo el comportamiento del agente, sumaria un ruido innecesario y nos desviaria del tema que elegimos de la consigna del proyecto para esta materia.

Lo que buscamos demostrar aquí es únicamente la capacidad matemática de nuestro índice vectorial **FAISS** para encontrar la información correcta: dada una alerta de Wazuh, ¿puede el sistema traer la técnica MITRE esperada entre sus primeros 5 resultados?

Para mantener el foco, el diseño de este experimento implica:

* `Aislar el Retriever:` Vamos a consultar directamente la base vectorial FAISS y la función de búsqueda por palabras clave, sin pasar por el Router ni ejecutar nodos del agente.

* `Ignorar la generación LLM:` No evaluaremos la redacción del reporte final ni las alucinaciones, solo validaremos los IDs de las técnicas (ej. T1059) recuperadas.

* `Métricas exactas:` Todo se reduce a tabular el **Hit@5** y el **MRR**, tal como solicita la corrección del profesor.

## 1. Conceptos clave

Antes de pasar a medir, debemos tener bien claro algunos conceptos:

Para garantizar que nuestro agente brinde diagnósticos precisos, Capazeta utiliza una arquitectura **RAG** (Retrieval-Augmented Generation). Este enfoque permite al modelo de lenguaje fundamentar sus respuestas buscando contexto técnico directamente en el catálogo **MITRE ATT&CK**. El motor que hace posible esta recuperación de información es **FAISS**, una herramienta que indexa nuestros datos y realiza búsquedas vectoriales para encontrar las técnicas semánticamente más similares a la alerta que estamos analizando.

Sin embargo, para validar que esta arquitectura realmente funciona, necesitamos medir la precisión de nuestro módulo de recuperación. Para ello, evaluaremos matemáticamente los resultados de FAISS utilizando dos métricas estándar en la industria. Por un lado, utilizaremos **Hit@K** (específicamente Hit@5) para responder a una pregunta binaria: ¿logró el sistema traer la técnica MITRE correcta dentro de sus primeros 5 resultados? Por otro lado, calcularemos el **RR** (Mean Reciprocal Rank) para evaluar la calidad del ordenamiento, verificando qué tan cerca del primer lugar apareció esa técnica correcta.

Finalmente, para automatizar este proceso, registrar los experimentos y comparar el rendimiento de nuestro índice vectorial contra una simple búsqueda por palabras clave, orquestaremos toda la evaluación utilizando **LangSmith**.

## 2. Setup

### Lang Smith

#### ¿Por qué utilizamos Lang Smith?
Para evaluar cuantitativamente nuestro sistema RAG, no basta con ejecutar consultas aisladas en la terminal. Necesitamos un entorno estructurado que nos permita medir el desempeño de forma repetible. Elegimos LangSmith para esta etapa por tres razones fundamentales:

* *Gestión del Ground Truth:* Nos permite cargar y versionar nuestro dataset de pruebas (las 20-30 alertas de Wazuh mapeadas manualmente a sus técnicas MITRE correctas).

* *Automatización de las Métricas:* Su motor de evaluación calcula nativamente el Hit@5 y el MRR ejecutando nuestras funciones de búsqueda iterativamente contra el dataset, evitando que codifiquemos bucles de cálculo manuales.

* *Trazabilidad y Análisis de Errores:* En los casos donde el sistema no logra un "Hit", la interfaz web de LangSmith captura la traza exacta de la ejecución, permitiéndonos inspeccionar qué documentos recuperó FAISS y entender por qué falló la similitud semántica.

A continuación, inicializamos el cliente y vinculamos nuestro notebook con la plataforma utilizando las variables de entorno de LangChain.

```bash
# LangSmith
LANGSMITH_TRACING=true
LANGSMITH_ENDPOINT=https://api.smith.langchain.com
LANGSMITH_API_KEY= # api key generada en interfaz de langsmith
LANGSMITH_PROJECT=IA
```

Ahora simplemente con esto podemos ir a la interfaz de lang smith y ver que vemos nuestro proyecto correctamente
<div align="center">
  <img src="./img/02_evaluacion_rag/image.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>

Tambien en la seccion de *tracing* podemos ver diferentes aspectos, desde latencia hasta el tag que propusimos que nos dice que modelo hemos usado

<div align="center">
  <img src="./img/02_evaluacion_rag/image copy.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>

## 3. Generacion del dataset (Ground Truth)

Lo que tenemos que hacer ahora es generar un dataset de prueba contra el cual vamos a contrastar los resultados del RAG y evaluarlo. Para hacer esto tendremos que hacer que se disparen una cierta cantidad de eventos que sean registrados en Wazuh y sean un mapeo claro a MITRE ATT&CK, por ejemplo realizaremos lo siguiente:

* `(T1110) Fuerza bruta SSH :` Intenta iniciar sesión por SSH varias veces con una contraseña incorrecta.

* `(T1046) Descubrimiento de red:` Ejecuta un escaneo con nmap localhost

* `(T1136) Creación de cuentas locales :` Añade un usuario de prueba ejecutando sudo useradd usuario_fantasma.

* `T1548 Abuso de privilegios :` Falla intencionalmente la autenticación de sudo o cambia permisos críticos de archivos en /etc (si tienes File Integrity Monitoring activado).

* `T1070 (Borrado de huellas):` Limpiar el historial de la terminal ejecutando history -c o intentando borrar un log con rm /var/log/auth.log.

* `T1053 (Tareas programadas):` Agregar un cron job persistente mediante crontab -e.

* `T1082 (Descubrimiento de sistema):` Ejecutar comandos de reconocimiento básicos como uname -a o cat /etc/os-release.

* `T1087 (Descubrimiento de cuentas):` Intentar leer el archivo de contraseñas protegidas con cat /etc/shadow.

* `T1049 (Descubrimiento de red):` Listar puertos y conexiones activas usando netstat -tunlp o ss -tuln.

* `T1222 (Modificación de permisos):` Cambiar los permisos de un archivo crítico a acceso público con chmod 777 /etc/passwd.

* `T1059 (Intérprete de comandos):` Ejecutar comandos ofuscados en base64 en la terminal.

### 3.1. Creacion de las alertas

Primero nos aseguramos que estemos reportando correctamente al SOC

<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 2.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

### 3.2. Configuracion en el agente

Antes de provocar estos eventos, debemos asegurarnos que nuestro agente este reportando todo correctamente, para esto debemos instalar lo siguiente
```bash
apt install auditd
```
Luego dentro de el archivo `/etc/audit/rules.d/wazuh.rules` le damos las siguientes instrucciones
```bash
-a always,exit -F arch=b64 -S execve -k audit-wazuh-c
-a always,exit -F arch=b32 -S execve -k audit-wazuh-c
-w /etc/shadow -p r -k audit-wazuh-r
-w /etc/passwd -p wa -k audit-wazuh-w
-w /etc/cron.d -p wa -k audit-wazuh-w
-w /etc/systemd/system -p wa -k audit-wazuh-w
-w /root/.bashrc -p wa -k audit-wazuh-w
```
Cargamos estas reglas con `augenrules --load` y verificamos con `auditctl -l`.

<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 9.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>

Luego en `/var/ossec/etc/ossec.conf` habilitamos la lectura
```bash
<localfile>
  <log_format>audit</log_format>
  <location>/var/log/audit/audit.log</location>
</localfile>

# y dentro de syschech
<directories realtime="yes" check_all="yes">/etc,/tmp</directories>
```

Luego reiniciamos el agente con `systemctl restart wazuh-agent`

### Script de generacion de alertas

Usaremos el siguiente script
<details>
<summary><strong>Ver el script generador completo (haz clic para expandir)</strong></summary>

```bash
#!/bin/bash
# Genera eventos benignos para Wazuh. Requiere root. Revierte lo que crea.
# Cada comando está mapeado a su táctica y técnica correspondiente de MITRE ATT&CK.

if [ "$EUID" -ne 0 ]; then
  echo "Ejecutar como root: sudo $0"
  exit 1
fi

FAKEUSER="usuario_fantasma_wazuh"
pausa() { sleep 3; }

echo "Generando eventos de seguridad para Wazuh..."

# ==========================================
# TÁCTICA: DISCOVERY (Descubrimiento)
# ==========================================

# [T1082] System Information Discovery: Obtener detalles del kernel y del SO
uname -a > /dev/null; cat /etc/os-release > /dev/null; pausa

# [T1033] System Owner/User Discovery: Identificar al usuario actual
whoami > /dev/null; id > /dev/null; pausa

# [T1057] Process Discovery: Listar procesos en ejecución
ps aux > /dev/null; pausa

# [T1083] File and Directory Discovery: Búsqueda de archivos de configuración
find /etc -maxdepth 2 -name "*.conf" > /dev/null 2>&1; pausa

# [T1087.001] Account Discovery (Local Account): Listar usuarios del sistema
cat /etc/passwd > /dev/null; getent passwd > /dev/null; pausa

# [T1049] System Network Connections Discovery: Listar puertos abiertos y conexiones
ss -tuln > /dev/null; pausa

# [T1016] System Network Configuration Discovery: Listar interfaces y rutas IP
ip addr > /dev/null; ip route > /dev/null; pausa

# [T1049 / T1016] ARP / Neighbor cache discovery
ip neigh > /dev/null; pausa

# [T1007] System Service Discovery: Listar servicios instalados en systemd
systemctl list-units --type=service > /dev/null 2>&1; pausa

# [T1518] Software Discovery: Listar paquetes instalados en el sistema
(dpkg -l || rpm -qa) > /dev/null 2>&1; pausa

# [T1046] Network Service Discovery: Escaneo de puertos locales con netcat
nc -vz localhost 1-1000 > /dev/null 2>&1; pausa


# ==========================================
# TÁCTICA: CREDENTIAL ACCESS (Acceso a Credenciales)
# ==========================================

# [T1003.008] OS Credential Dumping (/etc/shadow): Intentar leer hashes de contraseñas
cat /etc/shadow > /dev/null 2>&1; pausa

# [T1552.001] Unsecured Credentials (Files): Buscar contraseñas en texto plano
grep -rIl "password" /etc 2>/dev/null | head -n 5 > /dev/null; pausa

# [T1552.004] Unsecured Credentials (Private Keys): Búsqueda de llaves SSH privadas
find / -xdev \( -name "id_rsa" -o -name "id_ed25519" \) 2>/dev/null > /dev/null; pausa

# [T1110.001] Brute Force (Password Guessing): Intentos fallidos repetidos por SSH
for i in $(seq 1 10); do
  ssh -o BatchMode=yes -o StrictHostKeyChecking=no -o ConnectTimeout=3 \
      -o PreferredAuthentications=password usuario_inexistente@127.0.0.1 true > /dev/null 2>&1
done; pausa


# ==========================================
# TÁCTICAS: PERSISTENCE & PRIVILEGE ESCALATION (Gestión de Cuentas)
# ==========================================

# [T1136.001] Create Account (Local Account): Creación de un usuario local
useradd "$FAKEUSER" 2>/dev/null; pausa

# [T1098] Account Manipulation: Asignación de permisos de administrador al nuevo usuario
usermod -aG sudo "$FAKEUSER" 2>/dev/null || usermod -aG wheel "$FAKEUSER" 2>/dev/null; pausa

# [T1531] Account Access Removal: Eliminación de la cuenta y sus rastros
userdel -r "$FAKEUSER" 2>/dev/null; pausa


# ==========================================
# TÁCTICA: PERSISTENCE (Persistencia)
# ==========================================

# [T1053.003] Scheduled Task/Job (Cron): Persistencia mediante tarea programada
echo "* * * * * root /bin/true" > /etc/cron.d/wazuh_test; sleep 5
rm -f /etc/cron.d/wazuh_test; pausa

# [T1543.002] Create or Modify System Process (Systemd Service): Creación de servicio malicioso
cat > /etc/systemd/system/wazuh_test.service <<'EOF'
[Unit]
Description=Wazuh test service
[Service]
ExecStart=/bin/true
[Install]
WantedBy=multi-user.target
EOF
systemctl daemon-reload 2>/dev/null; sleep 5
rm -f /etc/systemd/system/wazuh_test.service
systemctl daemon-reload 2>/dev/null; pausa

# [T1546.004] Event Triggered Execution (Bash Profile/rc): Persistencia en .bashrc
cp ~/.bashrc /tmp/.bashrc.bak 2>/dev/null
echo "# wazuh_test" >> ~/.bashrc; sleep 3
mv /tmp/.bashrc.bak ~/.bashrc 2>/dev/null; pausa


# ==========================================
# TÁCTICA: PRIVILEGE ESCALATION (Escalada y Permisos)
# ==========================================

# [T1548.003] Abuse Elevation Control Mechanism (Sudo/Sudo Caching): Comprobación de privilegios
sudo -n -l > /dev/null 2>&1; sudo -n true > /dev/null 2>&1; pausa

# [T1222.002] File and Directory Permissions Modification (Linux): Asignar permisos 777
touch /tmp/wazuh_test_file.txt
chmod 777 /tmp/wazuh_test_file.txt; pausa


# ==========================================
# TÁCTICA: DEFENSE EVASION / FILE INTEGRITY
# ==========================================

# Generación de cambios en archivos para disparar el File Integrity Monitoring (FIM) de Wazuh
echo "test" > /etc/wazuh_test_fim.conf; sleep 5
echo "test2" >> /etc/wazuh_test_fim.conf; sleep 5
rm -f /etc/wazuh_test_fim.conf; pausa


# ==========================================
# TÁCTICA: EXECUTION & DEFENSE EVASION (Ejecución y Evasión)
# ==========================================

# [T1059.004] Command and Scripting Interpreter (Unix Shell) / [T1027] Obfuscated Files: Bash en base64
echo "d2hvYW1p" | base64 -d | bash > /dev/null; pausa

# [T1059.006] Command and Scripting Interpreter (Python) / [T1027]: Python ofuscado en base64
python3 -c "import base64;exec(base64.b64decode('cHJpbnQoMSk='))" > /dev/null 2>&1; pausa

# [T1140] Deobfuscate/Decode Files or Information: Decodificar payload en disco
echo "d2hvYW1p" > /tmp/wazuh_payload.b64
base64 -d /tmp/wazuh_payload.b64 > /dev/null; pausa

# [T1070.003] Indicator Removal on Host (Clear Command History): Borrar historial de la shell
cp ~/.bash_history /tmp/.bash_history.bak 2>/dev/null
: > ~/.bash_history; history -c
mv /tmp/.bash_history.bak ~/.bash_history 2>/dev/null; pausa
rm -f /tmp/wazuh_payload.b64 /tmp/wazuh_test_file.txt; pausa


# ==========================================
# TÁCTICAS: COLLECTION & COMMAND AND CONTROL
# ==========================================

# [T1560.001] Archive Collected Data (Archive via Utility): Comprimir datos para exfiltración
tar czf /tmp/wazuh_archive.tgz /etc/hostname > /dev/null 2>&1
rm -f /tmp/wazuh_archive.tgz; pausa

# [T1105] Ingress Tool Transfer: Descargar herramientas o payloads desde internet (C2)
curl -s -o /tmp/wazuh_dl.html https://example.com > /dev/null 2>&1 || \
  wget -q -O /tmp/wazuh_dl.html https://example.com > /dev/null 2>&1
rm -f /tmp/wazuh_dl.html

echo "Eventos generados. Revisa el dashboard de Wazuh."
```

El script se guardo dentro de `tests` y antes de ejecutarlo es necesario darle permisos de ejecucion: `chmod +x wazuh_generar_eventoss.sh` y ejecutarlo como `sudo`


<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 10.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>



### 3.3. Export al csv

Vamos a la seccion de ``Threats Hunting > Events`` y vemos que efectivamente se han captado los logs que queriamos. Ahora tenemos que elegir los correctos y exportarlos en csv

<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 6.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>

Antes de pasar a hacerlo, podemos ir a la vista de clasificacion de alertas en MITRE&ATACK del propio Wazuh para constatar cuantas de estas alertas pudo clasificar correctamente. Vemos que ha podido hacerlo unicamente con 4 alertas, por lo que justifica lo que estamos haciendo con el agente

<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 5.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>

Podemos decir que el valor agregado de Capazeta frente a un SIEM convencional como Wazuh se resume en tres puntos principales:

* `Rigidez vs. Flexibilidad Semántica:` Wazuh clasifica eventos mediante reglas estáticas en XML. Si una regla detecta un evento (como un bloqueo de AppArmor o un cambio de puerto) pero el creador de la regla no le escribió explícitamente la etiqueta <mitre><id>TXXXX</id></mitre> en su configuración, el SIEM lo muestra como un evento "sin técnica MITRE". Capazeta resuelve esto analizando el significado en lenguaje natural del log mediante embeddings, sin depender de etiquetas preexistentes.

* `Cobertura de Alertas Genéricas:` En entornos de producción, la gran mayoría de los logs del sistema operativo (eventos de kernel, auditd, autenticaciones PAM) son genéricos. Wazuh los registra como nivel de severidad bajo o sin contexto de amenaza. Nuestro RAG es capaz de inferir qué técnica del catálogo MITRE ATT&CK está teniendo lugar aunque la alerta no haya sido diseñada específicamente para ese ataque.

* `Reducción del esfuerzo manual:` Evita que el equipo de SOC (Security Operations Center) tenga que mantener y actualizar manualmente miles de reglas XML de mapeo a medida que el catálogo de MITRE evoluciona.


Para visualizar mejor nuestras alertas generadas es mejor hacer el export desde `Explore>Visualitation`
<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 11.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>

Luego de darle al boton de `export formatted`

Miremos ahora la salida que nos da
<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 12.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

Nuestro trabajo ahora es clasificar estas alertas a su regla MITRE&TTACK determinada

### 3.4. Clasificacion de reglas

Sobre el excel vamos a crear las columnas que creamos correspondiente donde describamos su mapeo a regla de MITRE&TTACK

<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 13.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>

Esto nos sirve a nosotros para entender su equivalente a la regla, pero para cargarselo a Lang smith y clasificar su desempeno solo nos hace falta dos columnas: 

`rule.description` y `mitre.id`. Esta decisión de diseño responde directamente al funcionamiento de los motores de búsqueda semántica y a los requisitos del framework de evaluación:

* `rule.description(Input / Consulta):` Actúa como la entrada en lenguaje natural que el módulo **RAG** debe procesar. Dado que la recuperación de información (tanto en FAISS como en el baseline de palabras clave) se basa en el significado semántico o léxico del texto, se requiere la descripción cruda del evento. Metadatos operativos como el ID de la regla (*rule.id*), el nivel de severidad o el timestamp carecen de valor semántico para el cálculo de embeddings y solo introducirían ruido en el espacio vectorial.

* `mitre.id (Output / Ground Truth):` Constituye la etiqueta de referencia. LangSmith utiliza este identificador para contrastar los documentos recuperados por el sistema RAG y determinar si la técnica correcta de MITRE ATT&CK fue encontrada. Esta columna es indispensable para el cálculo automatizado de las métricas de rendimiento del recuperador, específicamente Hit@5 y MRR (Mean Reciprocal Rank).

En conclusión, esta estructura minimalista aísla la variable que el experimento busca medir: la capacidad del sistema para inferir y mapear correctamente una táctica de amenaza basándose de forma exclusiva en la descripción semántica de un evento de seguridad aislado.

<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 14.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

Ambos csv van a estar en la carpeta de `test`, para entender mejor el mapeo se recomienda leer el `wazuh_ground_truth_annoted.csv`, ya que esta con mas descripciones utiles, sin embargo para testear usaremos el `dataset_evaluacion_langsmith.csv`

## 4. Evaluacion de performance del RAG

### 4.1. Subida a la nube de Lang Smith

Necesitamos enviar los pares de datos (alerta -> técnica) a la nube de LangSmith para que queden registrados como nuestra **Ground Truth**.

In [7]:
import pandas as pd
from langsmith import Client

# Inicializamos el cliente (debemos tener las variables de entorno LANGCHAIN_API_KEY configuradas)
client = Client()
dataset_name = "Evaluacion_Wazuh_MITRE"

# Cargamos el CSV de evaluación
df_eval = pd.read_csv("../tests/dataset_evaluacion_langsmith.csv")

# Creamos el dataset en el proyecto
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Dataset de 25 alertas únicas de Wazuh para evaluar clasificación MITRE ATT&CK."
)

# Iteramos sobre el DataFrame para subir los ejemplos (inputs y outputs)
for _, row in df_eval.iterrows():
    client.create_example(
        inputs={"rule.description": row["rule.description"]},
        outputs={"mitre_id": row["mitre_id"]},
        dataset_id=dataset.id
    )
    
print(f"Dataset '{dataset_name}' creado con {len(df_eval)} ejemplos.")

Dataset 'Evaluacion_Wazuh_MITRE' creado con 31 ejemplos.


Para comprobar que se ha subido correctamente a Lang Smith, vamos a la seccion de *Dataset and Experiments*


<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 15.png" width="700" alt="Prueba de agentes conectados en consola" />
</div>

Vemos que se ha creado correctamente la base de datos que acabamos de subir

### 4.2. Definición del Baseline por Palabras Clave

Para demostrar que el RAG aporta valor, necesitamos el punto de comparación. Esta función implementará una búsqueda tradicional usando TF-IDF sobre la base de datos de MITRE.

**TF-IDF (Term Frequency - Inverse Document Frequency)** es un algoritmo estadístico de búsqueda léxica que mide la importancia relativa de una palabra dentro de un documento en relación con una colección completa de textos (corpus).

    TF (Term Frequency): Mide la frecuencia con la que aparece una palabra en un texto específico.

    IDF (Inverse Document Frequency): Penaliza las palabras demasiado comunes en todo el conjunto de datos (ej. "system", "error", "log") y resalta aquellas más raras y distintivas (ej. "rootkit", "shadow").

Al combinar ambos valores, TF-IDF genera una representación numérica del texto basada exclusivamente en el recuento y peso de sus palabras exactas.

#### ¿Por qué lo comparamos contra el RAG?

La comparación entre TF-IDF y el motor RAG de Capazeta (FAISS + Embeddings) responde a tres razones clave en la metodología del proyecto:

* Establecer una Línea de Base (Baseline): En investigación y desarrollo de IA, no es posible afirmar que una arquitectura RAG es efectiva sin compararla contra un método tradicional sin IA. TF-IDF representa el estándar clásico de búsqueda por palabras clave.

* Evidenciar el contraste entre Búsqueda Léxica y Búsqueda Semántica:

    * TF-IDF (Léxico): Depende del vocabulario explícito. Si la alerta de Wazuh dice "New user added to the system", TF-IDF no logra asociarla a la técnica T1136.001 (Create Account) porque no comparten las mismas palabras.

    * RAG (Semántico): Entiende la intención y el contexto de la alerta en un espacio vectorial multidimensional, logrando mapear eventos independientemente de la sintaxis o los sinónimos utilizados.


In [ ]:
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Cargaoms la base de conocimiento de MITRE desde el JSON
json_path = "../data/mitre/techniques.json" 

with open(json_path, 'r', encoding='utf-8') as f:
    mitre_data = json.load(f)

ids_mitre = []
textos_mitre = []

# Procesamiento exacto para  estructura de JSON
for t_id, info in mitre_data.items():
    ids_mitre.append(t_id)
    nombre = info.get("name", "")
    descripcion = info.get("description", "")
    textos_mitre.append(f"{nombre} {descripcion}")

print(f"Se cargaron {len(ids_mitre)} técnicas de MITRE exitosamente.")
# 2. Inicializamos y ajustamos el modelo TF-IDF (El motor de palabras clave)
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(textos_mitre)

# 3. Definimos el pipeline del Baseline para LangSmith
def keyword_baseline_pipeline(inputs: dict) -> dict:
    """
    Busca las 5 técnicas de MITRE más similares a la alerta usando coincidencia exacta de palabras (TF-IDF).
    """
    alerta_texto = inputs["rule.description"]
    
    # Vectorizamos la alerta entrante
    alerta_vec = vectorizer.transform([alerta_texto])
    
    # Calculamos la similitud contra todas las técnicas de MITRE
    similitudes = cosine_similarity(alerta_vec, tfidf_matrix).flatten()
    
    # Obtenemos los índices de los 5 resultados con mayor puntuación
    top_5_indices = similitudes.argsort()[-5:][::-1]
    
    # Extraemos los IDs correspondientes
    tecnicas_recuperadas = [ids_mitre[i] for i in top_5_indices]
    
    return {"tecnicas_recuperadas": tecnicas_recuperadas}

# Prueba rápida manual antes de ejecutar LangSmith:
ejemplo_alerta = {"rule.description": "Successful sudo to ROOT executed"}
print("Predicción Baseline:", keyword_baseline_pipeline(ejemplo_alerta))

Se cargaron 867 técnicas de MITRE exitosamente.
Predicción Baseline: {'tecnicas_recuperadas': ['T1548.003', 'T1206', 'T1130', 'T1553.004', 'T1160']}


### 4.3. Funcion de evaluacion

In [ ]:
from langsmith.evaluation import evaluate
def mitre_retrieval_evaluator(run, example) -> list[dict]:
    """
    Evaluador robusto a prueba de errores de parseo o tipos de datos en LangSmith.
    """
    try:
        # 1. Extraer la técnica esperada de forma segura
        ref_out = example.outputs.get("mitre_id") or example.outputs.get("reference.mitre_id")
        if isinstance(ref_out, dict):
            tecnica_esperada = str(ref_out.get("mitre_id", "")).strip()
        else:
            tecnica_esperada = str(ref_out).strip()
            
        # 2. Extraer las técnicas recuperadas por la corrida
        outputs = run.outputs or {}
        tecnicas_recuperadas = outputs.get("tecnicas_recuperadas", [])
        
        # Sanitizar lista a strings simples
        tecnicas_recuperadas = [str(t).strip() for t in tecnicas_recuperadas]
        
        # 3. Calcular Hit@5
        hit = 1.0 if tecnica_esperada in tecnicas_recuperadas else 0.0
        
        # 4. Calcular MRR
        if tecnica_esperada in tecnicas_recuperadas:
            rank = tecnicas_recuperadas.index(tecnica_esperada) + 1
            mrr = 1.0 / rank
        else:
            mrr = 0.0

        return [
            {"key": "hit@5", "score": hit},
            {"key": "mrr", "score": mrr}
        ]

    except Exception as e:
        # Si ocurre cualquier falla inesperada, devuelve 0.0 en lugar de dejar la fila vacía (No feedback)
        return [
            {"key": "hit@5", "score": 0.0},
            {"key": "mrr", "score": 0.0}
        ]
    
# 2. Ejecutar la Evaluación en LangSmith
dataset_name = "Evaluacion_Wazuh_MITRE"


### 4.4. Evaluacion de TF-IDF

In [6]:

print("Iniciando evaluación del Baseline por Palabras Clave (TF-IDF)...")

results_baseline = evaluate(
    keyword_baseline_pipeline,
    data=dataset_name,
    evaluators=[mitre_retrieval_evaluator],
    experiment_prefix="Baseline_Keywords_TFIDF",
)
df_tf_idf = results_baseline.to_pandas()
df_tf_idf.to_csv(f"../reports/evaluacion_capazeta_tf_idf.csv", index=False)
print("¡Evaluación finalizada! Revisa tu panel de LangSmith.")

Iniciando evaluación del Baseline por Palabras Clave (TF-IDF)...
View the evaluation results for experiment: 'Baseline_Keywords_TFIDF-b660a830' at:
https://smith.langchain.com/o/954863b9-6bd1-4509-9100-6980d0c6d6e0/datasets/2415f52e-806c-4939-b514-adada324b15c/compare?selectedSessions=0dd86ec1-546f-48c8-8e3b-6c38eaf4fc8c


¡Evaluación finalizada! Revisa tu panel de LangSmith.


Si visitamos el link vamos a ver lo siguiente

<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 24.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>

Nos interesa particularmente la siguiente imagen, la cual nos muestra que obtuvo el puntaje de Hit rate de 0.32, el cual podemos decir que es muy bajo


<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 17.png" width="200" alt="Prueba de agentes conectados en consola" />
</div>

Observando la pagina extraemos lo siguiente:

##### *Métricas Generales del Baseline (TF-IDF)*

  * Hit@5: 32.26% (10 aciertos de 31).

  * MRR (Mean Reciprocal Rank): 0.2769 (~0.28).

  * Latencia Promedio: ~6.8 ms por consulta.

##### *Análisis de Patrones de Comportamiento*

* `Fortalezas:` Extremadamente rápido y preciso en alertas con coincidencia léxica directa o términos técnicos únicos. Por ejemplo, en "Auditd: Configuration changed." obtuvo una coincidencia exacta en posición #1 (MRR 1.0) para la técnica T1562.012, al igual que en "Possible kernel level rootkit" (T1014).

* `Debilidades:` Sufre una degradación crítica frente a variaciones de vocabulario, paráfrasis o descripciones operativas. No logra asociar "New user added to the system" con T1136.001 (Create Account) ni "sshd: authentication failed" con T1110.001 (Password Guessing), debido a que carecen de palabras compartidas en el vocabulario léxico.

### 4.5. Evaluacion de RAG


In [2]:
import os
import sys
from pathlib import Path

# El kernel corre en notebooks/: el índice FAISS vive en la raíz del repo.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.environ["MITRE_DATA_DIR"] = str(root / "data" / "mitre")

# Si el kernel ya importó capazeta con otra ruta o con otro modelo de embeddings,
# forzamos reimportar (config, embeddings y rag) para no mezclar dimensiones.
for _mod in [
    m
    for m in sys.modules
    if m in {"capazeta.config", "capazeta.embeddings"} or m.startswith("capazeta.rag")
]:
    del sys.modules[_mod]

from capazeta.rag import search


In [3]:
from capazeta.config import EMBEDDING_MODEL

slug = EMBEDDING_MODEL.split("/")[-1]


def rag_capazeta_pipeline(inputs: dict) -> dict:
    """Consulta el índice FAISS de Capazeta y devuelve el Top 5 de técnicas."""
    alerta_texto = inputs["rule.description"]
    resultados = search(alerta_texto, k=5)
    return {"tecnicas_recuperadas": [r["id"] for r in resultados if r.get("id")]}


results_rag = evaluate(
    rag_capazeta_pipeline,
    data=dataset_name,
    evaluators=[mitre_retrieval_evaluator],
    experiment_prefix=f"Capazeta_RAG_FAISS_{slug}",
)

df_rag = results_rag.to_pandas()
df_rag.to_csv(f"../reports/evaluacion_capazeta_rag_{slug}.csv", index=False)
print(f"¡Evaluación finalizada con {EMBEDDING_MODEL}! Revisa tu panel de LangSmith.")

View the evaluation results for experiment: 'Capazeta_RAG_FAISS_gemini-embedding-001-4642950a' at:
https://smith.langchain.com/o/954863b9-6bd1-4509-9100-6980d0c6d6e0/datasets/2415f52e-806c-4939-b514-adada324b15c/compare?selectedSessions=f80c548b-bb60-4a0f-a37e-4c1b4d330d3b


¡Evaluación finalizada con models/gemini-embedding-001! Revisa tu panel de LangSmith.


Observemos ahora el resultado
<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 25.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>

##### Métricas de Desempeño (Capazeta RAG)

  * Hit@5: 67.74% (21 aciertos de 31).

  * MRR (Mean Reciprocal Rank): 0.4866 (~0.49).

  * Latencia Promedio: ~636.4 ms por consulta.

##### Analisis de comportamiento

* `Fortalezas:` Captura la intención semántica de la alerta en lugar de depender de palabras exactas. Resolvió exitosamente 12 alertas donde TF-IDF falló completamente, logrando ubicar la respuesta en la primera posición (MRR 1.0) para eventos como "Common web attack." (T1190), "Audit: Deleted: /tmp/." (T1070.004), e identificando T1136.001 en "New user added to the system.".

* `Debilidades` (32.26% de fallos):

  * Confusión Acción vs. Intención: En "Audit: Watch - Write access: /etc/passwd.", la búsqueda vectorial favoreció técnicas de modificación de permisos (T1222) y credenciales (T1003.008) en lugar de creación de cuentas (T1136.001).

  * Falta de contexto de impacto: En "Wazuh agent stopped.", el modelo recuperó interrupción de servicio (T1489) en lugar de evasión de defensas (T1562.001).

### 4.6. Analisis comparativo

| **Métrica / Aspecto**       | **Baseline (TF-IDF)** | **Capazeta RAG (Gemini)** | **Diferencia / Impacto**                                  |
|-----------------------------|----------------------|---------------------------|----------------------------------------------------------|
| **Hit@5 (Tasa de Acierto)** | 32.26% (10/31)       | **67.74% (21/31)**         | **+35.48%** (Se duplica la tasa de acierto)              |
| **MRR (Ranking)**           | 0.2769               | **0.4866**                 | **+0.2097** (Mejora del 75% en la posición del resultado) |
| **Latencia Promedio**       | **6.8 ms**           | 636.4 ms                   | +629.6 ms (Costo computacional de la API vectorial)      |

El análisis de la tabla comparativa revela que la transición de un modelo de búsqueda léxica (TF-IDF) a una arquitectura semántica (Capazeta RAG con embeddings de Gemini) impacta profundamente en la calidad de la clasificación de alertas de Wazuh hacia el framework MITRE ATT&CK.   

* **Salto Cuantitativo en Efectividad (Hit@5):** El modelo RAG logra recuperar la técnica correcta en el Top 5 en un 67.74% de los casos (21 de 31), superando ampliamente el magro 32.26% (10 de 31) del Baseline. Esta mejora del +35.48% (que prácticamente duplica la tasa de acierto) demuestra que el sistema semántico logra asociar con éxito el contexto operativo de una alerta con su intención táctica, superando la limitación de TF-IDF que dependía estrictamente de coincidencias exactas de vocabulario.   

* **Mejora en la Precisión del Ranking (MRR):** El incremento del MRR de 0.2769 a 0.4866 refleja una mejora del 75% en el posicionamiento del resultado correcto. Esto significa que el agente RAG no solo encuentra la técnica correcta con mayor frecuencia, sino que tiende a ubicarla sistemáticamente en los primeros puestos (típicamente entre la posición 1 y 2). Para un analista de SOC, esto se traduce en una drástica reducción de la carga cognitiva al evaluar los falsos positivos.   

* **El Costo Computacional (Latencia):** El trade-off evidente de implementar IA generativa se observa en la latencia. Mientras TF-IDF resuelve la consulta de manera casi instantánea (6.8 ms), el procesamiento vectorial a través de la API de Gemini eleva el tiempo promedio a 636.4 ms. Sin embargo, en el contexto de enriquecimiento de alertas de ciberseguridad, un tiempo de respuesta de ~0.6 segundos es ampliamente aceptable y no interfiere con el flujo de trabajo de respuesta a incidentes, justificando con creces la inversión en tiempo a cambio del salto cualitativo en precisión.

#### Conclusion

La evaluación constata que el enfoque semántico basado en RAG es funcional, superior y necesario para este caso de uso. Mientras que los enfoques tradicionales se ven limitados por la disparidad lingüística entre los logs crudos y la nomenclatura formal de MITRE, Capazeta RAG demuestra tener la capacidad de "comprender" el evento de seguridad subyacente, consolidándose como una herramienta viable para automatizar y agilizar el Threat Hunting en un entorno de producción.

### 4.7. Analisis de resultados en Lang Smith

En la seccion de comparacion de experimentos podemos ver lo que estamos diciendo, donde el rosa corresponde al RAG con el modelo de generacion de embeddings de gemini y el verde a la baseline definida
<div align="center">
  <img src="./img/02_evaluacion_rag/image copy 26.png" width="1200" alt="Prueba de agentes conectados en consola" />
</div>

Esto nos da una alternativa mas amigable para poder ver sus diferencias, dandonos graficos mas interactivos.

El reporte de comparacion se encuentra en `reportes/` y se llama `comparacion_TDIF_vs_RAG.csv`